# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library. The steps below guide you through loading the Croissant metadata, examining record sets, extracting data, and performing basic exploratory analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs referenced by their `@id`.

First, we list all the record sets, then for each record set, we list its fields and columns (with their `@id`).

In [ ]:
# List all available record sets and their fields/columns by @id
print("Available RecordSets and their fields:")
record_sets = list(dataset.metadata.record_sets)
for record_set in record_sets:
    print(f"- RecordSet: {record_set['@id']}  (name: {record_set.get('name', '')})")
    fields = [f for f in record_set.get('fields', [])]
    if fields:
        for field in fields:
            print(f"    - Field: {field['@id']}  (name: {field.get('name', '')})")
            # List columns inside the field, if present
            if 'columns' in field:
                for col in field['columns']:
                    print(f"        - Column: {col['@id']}  (name: {col.get('name', '')})")
    else:
        print("    (No fields declared in this RecordSet)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the `@id`s from the above overview. Adjust the example below to match one of the record set IDs (e.g. `'cr:RecordSet/mental_health_survey'`) and its associated fields.

In [ ]:
# List all record set @id values
all_record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Record Set IDs:', all_record_set_ids)

# For this analysis, pick the main survey record set (adjust as needed if the actual ID differs):
main_record_set_id = all_record_set_ids[0]  # e.g., 'cr:RecordSet/mental_health_survey'

dataframes = {}
# Load all record sets into dataframes
for record_set_id in all_record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows with columns: {df.columns.tolist()}")

# Explore columns from the main table
main_df = dataframes[main_record_set_id]
print('Main DataFrame columns:', main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—filter numeric fields, normalize, and group by key attributes. All fields should be referenced by their `@id`.

We'll demonstrate filtering on a mental health score, normalization, and grouping by demographic field.

In [ ]:
# Identify numeric fields (by their @id) for demonstration
candidate_numeric_fields = [col for col in main_df.columns if main_df[col].dtype.kind in 'fi']
print("Numeric fields detected:", candidate_numeric_fields)

# Choose a numeric mental health indicator field for filtering (adjust if field IDs differ)
# For example, use '@id': 'cr:Field/gad7_score' for GAD-7 (Generalized Anxiety Disorder) score
numeric_field_id = candidate_numeric_fields[0] if candidate_numeric_fields else None
if not numeric_field_id:
    raise ValueError('No numeric field detected for demonstration.')

threshold = main_df[numeric_field_id].mean()  # Use mean as demo threshold
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value)")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nFirst rows of normalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field (using @id, e.g., 'cr:Field/gender')
candidate_group_fields = [col for col in main_df.columns if 'gender' in col.lower() or 'age' in col.lower() or 'education' in col.lower()]
group_field_id = None
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f'mean_{numeric_field_id}')
    print(f"\nGrouped {numeric_field_id} by {group_field_id} (mean values):")
    print(grouped_df)
else:
    print("No group field like 'gender', 'age', or 'education' found for grouping.")

## 5. Visualization
Visualize the selected numeric mental health score distribution by demographic attribute (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if group_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} distribution by {group_field_id}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    # Just plot the distribution
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, inspect, and analyze the FAIR² Mental Health Survey Dataset for Kilifi County, Kenya. You can extend the analysis by exploring other record sets, fields, or performing advanced modeling and data visualizations.

All entities—record sets, fields, columns—should always be referenced by their Croissant `@id` (see overview).